In [1]:
"""
================================================================================
PROJECT STEP 1: NOMINAL GROUND TRUTH EXTRACTION (STATIC PHASE)
================================================================================
DESCRIPTION:
This module establishes the core mathematical anchor points [A_nominal] and 
[B_nominal] for an open-loop unstable Continuous Stirred Tank Reactor (CSTR). 
We reject low-fidelity finite-difference approximations in favor of an exact 
analytical vector field mapping technique. This modular data pipeline separates
the hardware logging from matrix processing using object-oriented classes.

HARDWARE & EQUIPMENT SPECIFICATIONS:
- Primary Sensor Array: Integrated dual-channel inline sensor tracking a 2D 
  continuous state vector x(t) = [C_A(t), T(t)]^T.
  * Channel 1 (Concentration): High-precision inline spectrophotometer.
    Operational Bounds: [0.0, 2.0] kmol/m^3. Sampling rate: 100 Hz (dt = 0.01s).
  * Channel 2 (Temperature): Industrial Class-A Resistance Temperature Detector (RTD).
    Operational Bounds: [300.0, 500.0] K. Sampling rate: 100 Hz (dt = 0.01s).
- Actuator Assembly (u): Electronically controlled pneumatic coolant jacket valve.
  The input manifold u(t) dictates coolant temperature T_c(t).
  Physical Hardware Bounds: [250.0, 450.0] K.
================================================================================
"""

import numpy as np

class Initialization:
    """
    Class 1: Handles plant parameter storage, physical ODE evaluations, 
    and simulates the real-time sensor streaming phase under input dither excitation.
    """
    def __init__(self):
        # 1. Hardware Sensor Setup
        self.dt = 0.01          # 100 Hz hardware sensor sampling frequency
        self.n_samples = 1000   # Number of continuous snapshots collected
        
        # 2. Plant Physical Ground-Truth Constants
        self.q_V = 1.0          # Volumetric space velocity (q/V) [s^-1]
        self.C_Af = 1.0         # Feed concentration of reactant A [kmol/m^3]
        self.T_f = 350.0        # Feed temperature [K]
        self.k_0 = 1e8          # Arrhenius pre-exponential kinetic constant [s^-1]
        self.E_R = 6000.0       # Activation energy over gas constant (E/R) [K]
        self.dH_term = 2e5      # Dimensionless adiabatic heat of reaction term [K*m^3/kmol]
        self.UA_term = 0.5      # Jacket heat transfer coefficient term [s^-1]
        
        # 3. Unstable Steady-State Target Equilibrium (The Local Origin)
        self.C_A_ss = 0.5       # Steady-state concentration [kmol/m^3]
        self.T_ss = 400.0       # Steady-state temperature [K]
        self.T_c_ss = 350.0     # Steady-state nominal coolant jacket temperature [K]import numpy as np

    def cstr_nonlinear_dynamics(self, C_A, T, T_c):
        """
        Evaluates the exact, non-linear physical ordinary differential equations 
        (mass balance and energy balance) governing the internal material and energy 
        balances of the reactor.
        """
        # Arrhenius rate law expression
        reaction_rate = self.k_0 * np.exp(-self.E_R / T) * C_A
        
        # Mass Balance: d(C_A)/dt
        dC_A = self.q_V * (self.C_Af - C_A) - reaction_rate
        
        # Energy Balance: d(T)/dt
        dT = self.q_V * (self.T_f - T) + self.dH_term * reaction_rate - self.UA_term * (T - T_c)
        
        return np.array([dC_A, dT])

    def stream_sensor_data(self):
         """
        Responsible for simulating the open loop system.
        """
    np.random.seed(42)
    
    # 1. Pre-allocate state and control trajectories over time
    C_A = np.zeros(self.n_samples)
    T = np.zeros(self.n_samples)
    T_c = np.zeros(self.n_samples)
    
    X_dot_analytical = np.zeros((2, self.n_samples))
    
    # 2. Set initial values (C_A0, T0)
    C_A[0] = 0.55
    T[0] = 405.0
    
    for k in range(self.n_samples - 1):
        # Compute excitation signal
        T_c[k] = self.T_c_ss + np.sin(k * 0.05) * 8.0 + np.random.normal(0, 0.2)
        
        # Evaluate current time derivatives: x_dot = [dC_A/dt, dT/dt]
        x_dot = self.cstr_nonlinear_dynamics(C_A[k], T[k], T_c[k])
        X_dot_analytical[:, k] = x_dot
        
        # Euler step: Vector update using derivatives
        C_A[k + 1] = C_A[k] + x_dot[0] * self.dt
        T[k + 1]   = T[k]   + x_dot[1] * self.dt
        
    # Final step derivative evaluation
    T_c[-1] = self.T_c_ss + np.sin((self.n_samples - 1) * 0.05) * 8.0 + np.random.normal(0, 0.2)
    X_dot_analytical[:, -1] = self.cstr_nonlinear_dynamics(C_A[-1], T[-1], T_c[-1])
    
    # Construct deviation matrices relative to target equilibrium
    X_deviations = np.vstack([C_A - self.C_A_ss, T - self.T_ss])
    U_deviations = np.vstack([T_c - self.T_c_ss])


class GetGroundTruth:
    """
    Class 2: Consumes the raw data streams from the Initialization pipeline,
    constructs the data-augmented matrix manifolds, and extracts the 
    nominal A and B matrices via linear least-squares regression.
    """
    def __init__(self, init_instance):
        self.plant = init_instance

    def compute_nominal_matrices(self):
        """
        Executes the data-augmented regression pipeline over the analytic 
        vector field snapshots to uncover the flat local linear grid.
        """
        # Fetch the active sensor and derivative streams from Class 1
        X, U, X_dot = self.plant.stream_sensor_data()
        
        # Data Augmentation: Stack State (X) and Control Input (U) into Matrix Omega
        # Dimensions: (3 x N_SAMPLES)
        Omega = np.vstack([X, U])
        
        # Execute Moore-Penrose Pseudoinverse to resolve: X_dot = [A | B] * Omega
        A_B_augmented = X_dot @ np.linalg.pinv(Omega)
        
        # Slice the augmented mapping into independent nominal matrices
        A_nominal = A_B_augmented[:, :2]
        B_nominal = A_B_augmented[:, 2:3]
        
        return A_nominal, B_nominal


# ==============================================================================
# ROUTINE EXECUTION AND STABILITY VERIFICATION
# ==============================================================================
if __name__ == "__main__":
    # 1. Instantiate the classes
    plant_setup = Initialization()
    truth_extractor = GetGroundTruth(plant_setup)
    
    # 2. Compute the exact ground-truth matrices
    A_nominal, B_nominal = truth_extractor.compute_nominal_matrices()
    
    print("====================================================================")
    print("STATIC PHASE COMPLETE: NOMINAL GROUND TRUTH MATRICES EXTRACTION")
    print("====================================================================")
    print("A_nominal:\n", A_nominal)
    print("\nB_nominal:\n", B_nominal)
    
    # 3. Perform eigenvalue decomposition to prove open-loop positive stability
    eigenvalues = np.linalg.eigvals(A_nominal)
    print("\nOpen-Loop Eigenvalues (System Exponents):", eigenvalues)
    print("Verification: Contains positive exponent =", any(eigenvalues > 0))
    print("====================================================================")

NameError: name 'self' is not defined

In [ ]:
# """
# ================================================================================
# PROJECT STEP 2: LYAPUNOV STABILITY & EQUILIBRIUM-ELLIPSOID BARRIER SYNTHESIS
# ================================================================================
# EQUILIBRIUM-CENTRIC SAFE SET DESIGN DOCUMENTATION:
#
# Rather than assuming arbitrary operational factors, the safe set C is rigorously 
# derived from the equilibrium points and structural energy states of the system itself.
# Following Nagumo's theorem and framework, the safe set is defined as a permanent 
# geometric cage built directly around the local origin (equilibrium state x_e = 0):
#
#     C = { x in R^n : x^T * P * x <= delta^2 }
#
# The corresponding Control Barrier Function is explicitly modeled as:
#
#     h(x) = delta^2 - x^T * P * x
#
# Where:
# - P is the positive-definite matrix defining the ellipsoidal shape of the safe zone.
# - delta^2 is the maximum allowable potential energy deviation before the system 
#   ruptures its boundary cage.
#
# ARCHITECTURAL DESIGN DISCLAIMERS:
#
# 1. OPTIMIZATION SOLVER SELECTION:
#    We implement a quadratic Lyapunov function V(x) = x^T*P*x paired with a 
#    control-affine system. Because our performance bowl and operator safety 
#    boundaries result in constraints that remain strictly LINEAR with respect to 
#    the control input matrix [u], the online optimization reduces to a standard 
#    convex Quadratic Program (QP). We can therefore utilize standard, lightweight 
#    numerical solvers (such as SciPy's SLSQP). 
#    CRITICAL DISCLAIMER: If we were to implement more complex, non-quadratic, or 
#    non-smooth Lyapunov functions (e.g., Sum-of-Squares polynomials), the constraint 
#    space would become highly non-linear or semi-definite. This would require 
#    advanced, dedicated convex optimization parsing packages like CVXPY paired with 
#    high-performance solvers (e.g., OSQP, ECOS, or SCS).
#
# 2. HIGHER-ORDER LIE DERIVATIVES AND SYSTEM CHATTERING:
#    In this formulation, the control input u alters the state derivative x_dot 
#    directly in the first step. Because B is mapped directly against the gradient 
#    of h(x), this barrier manifold has an exact Relative Degree of 1 (Lg_h != 0).
#    CRITICAL SYSTEMIC RISK: If this were a more complex system where the control 
#    input was structurally decoupled from the safety states by multiple layers of 
#    integrators, we would be forced to compute higher-order Lie derivatives 
#    (e.g., Lf^2_h, Lg_Lf_h) to find where the input appears. In physical plant 
#    deployments, calculating higher-order derivatives of noisy, incoming sensor 
#    streams acts as an extreme noise amplifier. This destabilizes the QP solver's 
#    decision boundary, causing rapid, high-frequency switching of the control 
#    signal known as ACTUATOR CHATTERING, which rapidly degrades physical valves 
#    and mechanical components.
# ================================================================================
# """
#
# import numpy as np
# from scipy.linalg import solve_continuous_lyapunov
#
# class ControlSynthesis:
#     """
#     Class 3: Consumes the ground-truth physical matrices (A, B) and nominal 
#     control laws to map the global Lyapunov stability and equilibrium-centered safety cages.
#     """
#     def __init__(self, A_nominal, B_nominal):
#         # Ground-Truth Physical State Space Vectors extracted from Step 1
#         self.A = A_nominal
#         self.B = B_nominal
#         
#         # SYNTHESIS GROUND TRUTH MATRIX (K):
#         # The gain matrix K is classified as a synthesis ground truth. While A and B 
#         # define the ground truth of the natural physics, K acts as the absolute mathematical 
#         # anchor point defining our nominal closed-loop performance intent. It is 
#         # mathematically impossible to construct the P ledger matrix without anchoring K first.
#         self.K = None          
#         self.P = None          # Lyapunov Tensor Shape Matrix (The Ledger of Tension)
#         self.delta_sq = None   # Maximum allowable energy boundary capacity (delta^2)
#
#     def define_lyapunov_function(self):
#         """
#         Executes the matrix operations to construct the Control Lyapunov Function bowl.
#         Formula: (A - B*K)^T * P + P * (A - B*K) = -Q
#         """
#         # Step 1: Establish the performance penalty ledger (Q)
#         Q = np.eye(2)
#         
#         # Step 2: Establish the Nominal Control Ground Truth (K)
#         # This matrix places the target closed-loop poles to stabilize the operating cage.
#         self.K = np.array([[4.0, 5.0]]) 
#         
#         # Step 3: Compute Closed-Loop State Transition Matrix
#         A_cl = self.A - self.B @ self.K
#         
#         # Step 4: Solve the Continuous-Time Algebraic Lyapunov Equation
#         # Generates the positive-definite matrix P defining the geometry of our bowl.
#         self.P = solve_continuous_lyapunov(A_cl.T, -Q)
#         
#         # Step 5: Derive Safe Set C Boundary dynamically from system eigenvalues
#         # We pull the open-loop eigenvalues and scale the allowable energy bound
#         # inversely proportional to the severity of the unstable thermal runaway branch.
#         eigenvalues = np.linalg.eigvals(self.A)
#         max_positive_exponent = np.max(eigenvalues[eigenvalues > 0])
#         
#         # delta^2: Maximum allowable deviation distance before system failure
#         self.delta_sq = 15.0 / max_positive_exponent 
#         
#         return self.P, self.K, self.delta_sq
#
#     def evaluate_lyapunov_stability(self, x, u_val):
#         """
#         Surveys the current coordinate to check how much the interference wave 
#         is compressing or expanding down the Lyapunov performance bowl.
#         Formula: V_dot = 2 * x^T * P * (A*x + B*u)
#         """
#         # Compute the continuous raw vector field matrix x_dot for this instant
#         x_dot = self.A @ x + self.B.flatten() * u_val
#         
#         # Calculate the scalar potential energy: V(x) = x^T * P * x
#         V = x.T @ self.P @ x
#         
#         # Calculate the temporal rate of change of the energy metric
#         V_dot = 2.0 * (x.T @ self.P @ x_dot)
#         
#         return V, V_dot
#
#     def evaluate_cbf(self, x):
#         """
#         Evaluates the safety barrier and explicitly computes the Lie Derivatives
#         to isolate natural system drift from actuator authority.
#         
#         Safety Manifold Definition (Nagumo Equilibrium Cage):
#         h(x) = delta_sq - x^T * P * x >= 0 
#         
#         Lie Derivative Formulations (SHOW WORK via Matrix Chain Rule):
#         dh_dx = -2 * x^T * P
#         Therefore:
#         - Lf_h = (dh_dx) * f(x) = -2 * x^T * P * (A * x)
#         - Lg_h = (dh_dx) * g(x) = -2 * x^T * P * B
#         """
#         # Step 1: Calculate the absolute scalar safety margin h(x)
#         h = self.delta_sq - (x.T @ self.P @ x)
#         
#         # Step 2: SHOW WORK - Compute the analytical gradient row matrix: dh/dx
#         # Size: Row Vector (1 x 2)
#         dh_dx = -2.0 * (x.T @ self.P)
#         
#         # Step 3: Isolate independent drift and control components of the vector field
#         # where x_dot = f(x) + g(x)u = Ax + Bu
#         f_x = self.A @ x
#         g_x = self.B
#         
#         # Step 4: SHOW WORK - Compute Analytical Lie Derivatives
#         # Lf_h tracks how the natural, unforced physical drift of the CSTR alters
#         # the system's position relative to the performance envelope edge.
#         # Matrix product chain: (1x2) @ (2x1) = scalar
#         Lf_h = dh_dx @ f_x
#         
#         # Lg_h maps the explicit authority your control valve possesses to squeeze
#         # or alter the instantaneous energy growth trajectory.
#         # Matrix product: (1x2) @ (2x1) = scalar
#         Lg_h = dh_dx @ g_x
#         
#         # Cast elements to floating-point scalars for the downstream QP loop
#         Lf_h_scalar = float(Lf_h)
#         Lg_h_scalar = float(Lg_h)
#         
#         return h, Lf_h_scalar, Lg_h_scalar
